In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [2]:
import os
print(os.listdir('/kaggle/input/imdb-dataset-of-50k-movie-reviews'))

['IMDB Dataset.csv']


In [3]:
import pandas as pd
df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.describe()

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [6]:
df.sentiment.value_counts()

,count
sentiment,
positive,25000
negative,25000


**Remove Punctuation**

In [7]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [10]:
df['review'] = df['review'].str.lower()

df['review'] = df['review'].apply(
    lambda x: x.translate(str.maketrans('', '', string.punctuation))
)

**Remove HTML**

In [11]:
import re

def remove_html(text):
    pattern = re.compile(r'<.*?>')
    
    return pattern.sub('',text)

In [12]:
df['review'] = df['review'].apply(remove_html)

**Remove URL**

In [13]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    
    return pattern.sub('', text)

In [14]:
df['review'] = df['review'].apply(remove_url)

**Remove Stop Words**

In [18]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = ENGLISH_STOP_WORDS

def remove_stopwords(text):
    tokens = text.split()
    filtered = [token for token in tokens if token not in stop_words]
    return ' '.join(filtered)

df['review'] = df['review'].apply(remove_stopwords)

**Tokenization and Lemmatization**

In [22]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [23]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(lemmatized)

df['review'] = df['review'].apply(preprocess_text)

**Encoding**

In [24]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

**Model Training**

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

In [26]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

**LSTM**

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [29]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 338s 668ms/step - accuracy: 0.8196 - loss: 0.4027 - val_accuracy: 0.8630 - val_loss: 0.3292
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 333s 666ms/step - accuracy: 0.8987 - loss: 0.2625 - val_accuracy: 0.8609 - val_loss: 0.3217
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 380s 663ms/step - accuracy: 0.9232 - loss: 0.2081 - val_accuracy: 0.8687 - val_loss: 0.3599
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 382s 663ms/step - accuracy: 0.9389 - loss: 0.1633 - val_accuracy: 0.8618 - val_loss: 0.3529
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 378s 656ms/step - accuracy: 0.9534 - loss: 0.1294 - val_accuracy: 0.8666 - val_loss: 0.4130


In [30]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (64, 200, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (64, 128)              │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (64, 64)               │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (64, 64)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (64, 1)                │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,259,717 (16.25 MB)

 Trainable params: 1,419,905 (5.42 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,839,812 (10.83 MB)

In [31]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 83ms/step - accuracy: 0.8694 - loss: 0.4028
Test Accuracy: 0.8694000244140625


**Predictions**

In [32]:
def predict_sentiment(review):
    sequence = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequence, maxlen=200)
    prediction = model.predict(padded_sequence)
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'

    return sentiment

In [33]:
predictions = model.predict(X_test)

predicted_labels = (predictions > 0.5).astype(int)

313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 84ms/step


In [34]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predicted_labels))

              precision    recall  f1-score   support

           0       0.88      0.85      0.87      4961
           1       0.86      0.89      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [35]:
model.save("sentiment_lstm.keras")

In [36]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)